In [1]:
# langgraph_command_tracker.py
# A LangGraph workflow that classifies, validates, and tracks file commands
# Uses: StateGraph, nodes, edges, compile — all properly implemented

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator
import json

# ── Step 1: Define State Schema ───────────────────────────────────────────────
# This is what LangGraph remembers across nodes
class CommandState(TypedDict):
    command: str                              # raw user input
    command_type: str                         # classified type
    is_valid: bool                            # validation result
    reason: str                               # why valid or invalid
    history: Annotated[list, operator.add]    # all commands processed so far

# ── Step 2: Define Nodes ──────────────────────────────────────────────────────
# Each node is a function that takes state and returns updated state

def classify_command(state: CommandState) -> dict:
    """
    Node 1: Classifies the command into one of 6 types.
    Uses keyword matching to identify the action.
    """
    command = state["command"].lower()

    if "create" in command:
        command_type = "create_file"
    elif "edit" in command:
        command_type = "edit_file"
    elif "append" in command:
        command_type = "append_file"
    elif "delete" in command:
        command_type = "delete_file"
    elif "rename" in command:
        command_type = "rename_file"
    elif "run" in command:
        command_type = "run_script"
    else:
        command_type = "unknown"

    print(f"🔍 Classified as: {command_type}")
    return {"command_type": command_type}


def validate_command(state: CommandState) -> dict:
    """
    Node 2: Validates whether the command is safe to execute.
    Checks for unknown type and dangerous patterns.
    """
    command = state["command"].lower()
    command_type = state["command_type"]

    # Reject unknown commands
    if command_type == "unknown":
        return {
            "is_valid": False,
            "reason": "Command type not recognized. Supported: create, edit, append, delete, rename, run."
        }

    # Reject dangerous patterns
    dangerous = ["system32", "../", "rm -rf", ".exe", "format"]
    for pattern in dangerous:
        if pattern in command:
            return {
                "is_valid": False,
                "reason": f"Dangerous pattern detected: '{pattern}'"
            }

    return {
        "is_valid": True,
        "reason": "Command is safe to execute."
    }


def log_command(state: CommandState) -> dict:
    """
    Node 3: Logs the command to history in state.
    LangGraph accumulates history across runs using Annotated[list, operator.add].
    """
    log_entry = {
        "command": state["command"],
        "type": state["command_type"],
        "valid": state["is_valid"],
        "reason": state["reason"]
    }

    print(f"📝 Logged: {log_entry}")
    return {"history": [log_entry]}


def handle_invalid(state: CommandState) -> dict:
    """
    Node 4: Handles invalid commands — logs them with rejection reason.
    """
    log_entry = {
        "command": state["command"],
        "type": state["command_type"],
        "valid": False,
        "reason": state["reason"]
    }

    print(f"❌ Rejected: {state['reason']}")
    return {"history": [log_entry]}


# ── Step 3: Routing Function ──────────────────────────────────────────────────
def route_after_validation(state: CommandState) -> str:
    """
    Decides which node to go to after validation.
    If valid → log_command
    If invalid → handle_invalid
    """
    if state["is_valid"]:
        return "log_command"
    else:
        return "handle_invalid"


# ── Step 4: Build the Graph ───────────────────────────────────────────────────
def build_graph() -> any:
    """
    Assembles the LangGraph workflow:
    classify → validate → (log or reject) → END
    """
    graph = StateGraph(CommandState)

    # Add nodes
    graph.add_node("classify_command", classify_command)
    graph.add_node("validate_command", validate_command)
    graph.add_node("log_command", log_command)
    graph.add_node("handle_invalid", handle_invalid)

    # Set entry point
    graph.set_entry_point("classify_command")

    # Add edges
    graph.add_edge("classify_command", "validate_command")

    # Conditional edge after validation
    graph.add_conditional_edges(
        "validate_command",
        route_after_validation,
        {
            "log_command": "log_command",
            "handle_invalid": "handle_invalid"
        }
    )

    # Both paths end at END
    graph.add_edge("log_command", END)
    graph.add_edge("handle_invalid", END)

    # Compile the graph
    return graph.compile()


# ── Step 5: Run the workflow ──────────────────────────────────────────────────
def process_command(app, command: str, history: list = []) -> CommandState:
    """
    Runs the compiled graph for a single command.
    Passes existing history so state accumulates.
    """
    initial_state = {
        "command": command,
        "command_type": "",
        "is_valid": False,
        "reason": "",
        "history": history
    }

    result = app.invoke(initial_state)
    return result


# ── Main Loop ─────────────────────────────────────────────────────────────────
def main():
    print("🤖 LangGraph Command Tracker")
    print("Classifies, validates, and tracks file commands")
    print("=" * 45)

    # Build and compile graph once
    app = build_graph()
    print("✅ LangGraph compiled successfully\n")

    history = []

    while True:
        command = input("\nEnter command (or 'history' or 'exit'): ").strip()

        if command.lower() == "exit":
            print("\n👋 Goodbye!")
            break

        if command.lower() == "history":
            print("\n📋 Command History:")
            print("-" * 40)
            if not history:
                print("No commands yet.")
            for i, entry in enumerate(history, 1):
                status = "✅" if entry["valid"] else "❌"
                print(f"{i}. {status} [{entry['type']}] {entry['command']}")
            continue

        if not command:
            continue

        # Process through LangGraph
        print()
        result = process_command(app, command, history)

        # Update history from state
        history = result["history"]

        # Show result
        status = "✅ Valid" if result["is_valid"] else "❌ Invalid"
        print(f"{status} — {result['reason']}")


if __name__ == "__main__":
    main()

C:\Users\SAIM\AppData\Local\Programs\Python\Python313\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


🤖 LangGraph Command Tracker
Classifies, validates, and tracks file commands
✅ LangGraph compiled successfully




Enter command (or 'history' or 'exit'):  edit



🔍 Classified as: edit_file
📝 Logged: {'command': 'edit', 'type': 'edit_file', 'valid': True, 'reason': 'Command is safe to execute.'}
✅ Valid — Command is safe to execute.



Enter command (or 'history' or 'exit'):  delete



🔍 Classified as: delete_file
📝 Logged: {'command': 'delete', 'type': 'delete_file', 'valid': True, 'reason': 'Command is safe to execute.'}
✅ Valid — Command is safe to execute.



Enter command (or 'history' or 'exit'):  history



📋 Command History:
----------------------------------------
1. ✅ [edit_file] edit
2. ✅ [delete_file] delete



Enter command (or 'history' or 'exit'):  exit



👋 Goodbye!
